# Scraping Financial Data

This notebook scrapes market data from yfinance and macro data from FRED, saving to market.csv and macros.csv respectively.

In [2]:
# Install required libraries if not already installed
%pip install datasets pandas yfinance pandas_datareader

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import libraries
import pandas as pd
from datasets import load_dataset
import yfinance as yf

c:\Users\pibrys\anaconda3\envs\SMWA2026\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Pull market data from yfinance
import yfinance as yf

start_date = '2024-06-01'  # Start 5 weeks before basetable to warm up rolling windows (30d max)
end_date = '2024-11-04'

tickers = {
    '^GSPC': 'SP500',
    'CL=F': 'Oil',
    '^VIX': 'VIX',
    '^TNX': 'TenYearBond',
    'DX-Y.NYB': 'USDIndex'
}

df_list = []
for ticker, name in tickers.items():
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    if 'Adj Close' in df.columns:
        df = df[['Adj Close']].rename(columns={'Adj Close': name})
    elif 'Close' in df.columns:
        df = df[['Close']].rename(columns={'Close': name})
    else:
        print(f"Warning: {ticker} has no price data.")
        continue
    df_list.append(df)

if df_list:
    market_df = pd.concat(df_list, axis=1)
    if isinstance(market_df.columns, pd.MultiIndex):
        market_df.columns = market_df.columns.get_level_values(0)
    market_df.to_csv('Storage/market.csv', index=True)
    print('Market data saved to Storage/market.csv')
    print(market_df.tail())
else:
    print('No market data fetched.')

Market data saved to Storage/market.csv
Price             SP500        Oil        VIX  TenYearBond    USDIndex
Date                                                                  
2024-10-28  5823.520020  67.379997  19.799999        4.278  104.320000
2024-10-29  5832.919922  67.209999  19.340000        4.274  104.279999
2024-10-30  5813.669922  68.610001  20.350000        4.266  103.989998
2024-10-31  5705.450195  69.260002  23.160000        4.284  103.980003
2024-11-01  5728.799805  69.489998  21.879999        4.361  104.279999


In [5]:
# Pull macro data from FRED
from pandas_datareader import data as pdr

fred_api_key = '981ac4d5555a2e7f975219ee2bc806a1'
start_date = '2024-06-01'  # Start earlier to pick up July 2024 macro releases
end_date = '2024-11-04'

series = {
    'DSPIC96': 'RealDisposableIncome',
    'GDPC1': 'RealGDP',
    'UNRATE': 'UnemploymentRate',
    'CPIAUCSL': 'CPIInflation',
    'UMCSENT': 'ConsumerSentiment'
}

fred_data = {}
for symbol, name in series.items():
    try:
        fred_data[symbol] = pdr.DataReader(symbol, 'fred', start_date, end_date, api_key=fred_api_key)
        print(f"Fetched {symbol}: {fred_data[symbol].shape[0]} rows")
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")

if fred_data:
    macros_df = pd.concat(fred_data.values(), axis=1)
    macros_df.columns = list(series.values())
    macros_df.to_csv('Storage/macros.csv', index=True)
    print('Macros data saved to Storage/macros.csv')
    print(macros_df.tail())
else:
    print('No macros data fetched.')

Fetched DSPIC96: 6 rows
Fetched GDPC1: 2 rows
Fetched UNRATE: 6 rows
Fetched CPIAUCSL: 6 rows
Fetched UMCSENT: 6 rows
Macros data saved to Storage/macros.csv
            RealDisposableIncome    RealGDP  UnemploymentRate  CPIInflation  \
DATE                                                                          
2024-07-01               17743.2  23478.570               4.2       313.569   
2024-08-01               17752.9        NaN               4.2       314.062   
2024-09-01               17769.9        NaN               4.1       314.732   
2024-10-01               17810.5  23586.542               4.1       315.631   
2024-11-01               17851.4        NaN               4.2       316.528   

            ConsumerSentiment  
DATE                           
2024-07-01               66.4  
2024-08-01               67.9  
2024-09-01               70.1  
2024-10-01               70.5  
2024-11-01               71.8  


## Summary

The notebook has successfully scraped market data from yfinance and macro data from FRED, saving to Storage/market.csv and Storage/macros.csv respectively.